# Neural Networks – 2nd Assignment

SVMs on CIFAR-10 for a binary task: animals vs vehicles. The hyperparameters are tuned with 5-fold GridSearchCV on the GPU (cuML).

The first cell compares the final models with k-NN, Nearest Centroid and an MLP with hinge loss, and the next cells are the grid searches for the linear, RBF and polynomial kernels.

In [1]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from tensorflow.keras.datasets import cifar10

import cuml
from cuml.svm import SVC as GPU_SVC
from cuml.svm import LinearSVC
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

print(f" Libraries Loaded. GPU Mode (cuML): {cuml.__version__}, PyTorch: {torch.__version__}")

# =============================================================================
# 1. DATA PREPARATION (labels -1 and 1)
# =============================================================================
print("\n--- 1. Preparing CIFAR-10 data ---")
(x_train_orig, y_train_orig), (x_test_orig, y_test_orig) = cifar10.load_data()

animals = [2, 3, 4, 5, 6, 7]
vehicles = [0, 1, 8, 9]

def convert_labels(y_orig):
    y_flat = y_orig.flatten()
    y_new = np.zeros_like(y_flat, dtype=np.float32)
    mask_animals = np.isin(y_flat, animals)
    mask_vehicles = np.isin(y_flat, vehicles)
    y_new[mask_animals] = -1.0
    y_new[mask_vehicles] = 1.0
    return y_new

y_train = convert_labels(y_train_orig)
y_test = convert_labels(y_test_orig)

# Flattening & Scaling
X_train = x_train_orig.reshape(x_train_orig.shape[0], -1).astype('float32') / 255.0
X_test = x_test_orig.reshape(x_test_orig.shape[0], -1).astype('float32') / 255.0

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Data: {X_train_scaled.shape}")
print(f"Testing Data:  {X_test_scaled.shape}")

# =============================================================================
# VISUALIZATION (TWO SEPARATE PLOTS)
# =============================================================================
def visualize_performance(model_name, y_true, y_pred, x_orig_imgs):
    """
    Prints the confusion matrix first and the examples below it.
    """
    print(f"\n Visualizing Results for: {model_name}")

    # --- PLOT 1: Confusion Matrix ---
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Animals', 'Vehicles'],
                yticklabels=['Animals', 'Vehicles'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # --- PLOT 2: Sample Predictions ---
    plt.figure(figsize=(15, 3))
    indices = np.random.choice(len(y_true), 5, replace=False)

    for i, idx in enumerate(indices):
        plt.subplot(1, 5, i + 1)
        plt.imshow(x_orig_imgs[idx])

        true_label = "Vehicle" if y_true[idx] == 1.0 else "Animal"
        pred_label = "Vehicle" if y_pred[idx] == 1.0 else "Animal"

        color = 'green' if true_label == pred_label else 'red'

        plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=10, fontweight='bold')
        plt.axis('off')

    plt.suptitle(f'{model_name} - Random Test Samples', fontsize=12)
    plt.show()

# =============================================================================
# 2. MLP DEFINITION (Hinge Loss)
# =============================================================================
class HingeMLP(nn.Module):
    def __init__(self, input_dim=3072, hidden_dim=100):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def run_mlp_experiment(X_tr, y_tr, X_te, y_te):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    test_ds = TensorDataset(torch.tensor(X_te), torch.tensor(y_te))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False)

    model = HingeMLP().to(device)
    optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

    print("   Training MLP (Hinge)...")
    start = time.time()

    for epoch in range(20):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(xb).squeeze()
            loss = torch.mean(torch.clamp(1 - yb * outputs, min=0))
            loss.backward()
            optimizer.step()

    train_time = time.time() - start

    model.eval()

    # Train Acc
    correct_tr = 0
    with torch.no_grad():
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb).squeeze()
            preds = torch.where(out > 0, 1.0, -1.0)
            correct_tr += (preds == yb).sum().item()
    train_acc = correct_tr / len(y_tr)

    # Test Acc & Preds
    all_preds = []
    correct_te = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb).squeeze()
            preds = torch.where(out > 0, 1.0, -1.0)
            correct_te += (preds == yb).sum().item()
            all_preds.append(preds.cpu().numpy())

    test_acc = correct_te / len(y_te)
    all_preds = np.concatenate(all_preds)

    print(classification_report(y_te, all_preds, target_names=['Animals', 'Vehicles'], digits=4))

    return train_acc, test_acc, train_time, all_preds

# =============================================================================
# 3. RUNNING THE EXPERIMENTS
# =============================================================================
results = []

print("\n---  STARTING EXPERIMENTS ---")

# --- 1. RBF SVM ---
print("\n" + "-"*50)
print("  RBF SVM")
print("-" * 50)
rbf_model = GPU_SVC(kernel='rbf', C=3.94, gamma=0.00056, cache_size=2000, class_weight='balanced')
start = time.time()
rbf_model.fit(X_train_scaled, y_train)
t_rbf = time.time() - start
preds_rbf = rbf_model.predict(X_test_scaled)
acc_tr_rbf = accuracy_score(y_train, rbf_model.predict(X_train_scaled))
acc_te_rbf = accuracy_score(y_test, preds_rbf)
print(classification_report(y_test, preds_rbf, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("RBF SVM", y_test, preds_rbf, x_test_orig)
results.append(['RBF SVM', acc_tr_rbf, acc_te_rbf, t_rbf])

# --- 2. Linear SVM ---
print("\n" + "-"*50)
print("  Linear SVM")
print("-" * 50)
lin_model = LinearSVC(C=0.0001, penalty='l2', loss='squared_hinge', verbose=False, class_weight='balanced')
start = time.time()
lin_model.fit(X_train_scaled, y_train)
t_lin = time.time() - start
preds_lin = lin_model.predict(X_test_scaled)
acc_tr_lin = accuracy_score(y_train, lin_model.predict(X_train_scaled))
acc_te_lin = accuracy_score(y_test, preds_lin)
print(classification_report(y_test, preds_lin, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("Linear SVM", y_test, preds_lin, x_test_orig)
results.append(['Linear SVM', acc_tr_lin, acc_te_lin, t_lin])

# --- 3. Poly SVM ---
print("\n" + "-"*50)
print("  Poly SVM")
print("-" * 50)
poly_model = GPU_SVC(kernel='poly', degree=3, C=0.75, gamma=0.0003, coef0=1.0, cache_size=2000, class_weight='balanced')
start = time.time()
poly_model.fit(X_train_scaled, y_train)
t_poly = time.time() - start
preds_poly = poly_model.predict(X_test_scaled)
acc_tr_poly = accuracy_score(y_train, poly_model.predict(X_train_scaled))
acc_te_poly = accuracy_score(y_test, preds_poly)
print(classification_report(y_test, preds_poly, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("Poly SVM", y_test, preds_poly, x_test_orig)
results.append(['Poly SVM', acc_tr_poly, acc_te_poly, t_poly])

# --- 4. MLP Hinge ---
print("\n" + "-"*50)
print("  MLP Hinge Loss")
print("-" * 50)
acc_tr_mlp, acc_te_mlp, t_mlp, preds_mlp = run_mlp_experiment(X_train_scaled, y_train, X_test_scaled, y_test)
# VISUALIZATION
visualize_performance("MLP (Hinge)", y_test, preds_mlp, x_test_orig)
results.append(['MLP (Hinge)', acc_tr_mlp, acc_te_mlp, t_mlp])

# --- 5. 1-Nearest Neighbor ---
print("\n" + "-"*50)
print("  1-Nearest Neighbor")
print("-" * 50)
knn1 = KNeighborsClassifier(n_neighbors=1, n_jobs=-1)
start = time.time()
knn1.fit(X_train_scaled, y_train)
preds_knn1 = knn1.predict(X_test_scaled)
t_knn1 = time.time() - start
acc_tr_knn1 = knn1.score(X_train_scaled[:5000], y_train[:5000])
acc_te_knn1 = accuracy_score(y_test, preds_knn1)
print(classification_report(y_test, preds_knn1, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("1-NN", y_test, preds_knn1, x_test_orig)
results.append(['1-NN', acc_tr_knn1, acc_te_knn1, t_knn1])

# --- 6. 3-Nearest Neighbor ---
print("\n" + "-"*50)
print("  3-Nearest Neighbor")
print("-" * 50)
knn3 = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
start = time.time()
knn3.fit(X_train_scaled, y_train)
preds_knn3 = knn3.predict(X_test_scaled)
t_knn3 = time.time() - start
acc_tr_knn3 = knn3.score(X_train_scaled[:5000], y_train[:5000])
acc_te_knn3 = accuracy_score(y_test, preds_knn3)
print(classification_report(y_test, preds_knn3, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("3-NN", y_test, preds_knn3, x_test_orig)
results.append(['3-NN', acc_tr_knn3, acc_te_knn3, t_knn3])

# --- 7. Nearest Centroid ---
print("\n" + "-"*50)
print("  Nearest Class Centroid")
print("-" * 50)
ncc = NearestCentroid()
start = time.time()
ncc.fit(X_train_scaled, y_train)
t_ncc = time.time() - start
preds_ncc = ncc.predict(X_test_scaled)
acc_tr_ncc = ncc.score(X_train_scaled, y_train)
acc_te_ncc = accuracy_score(y_test, preds_ncc)
print(classification_report(y_test, preds_ncc, target_names=['Animals', 'Vehicles'], digits=4))
# VISUALIZATION
visualize_performance("Nearest Centroid", y_test, preds_ncc, x_test_orig)
results.append(['Nearest Centroid', acc_tr_ncc, acc_te_ncc, t_ncc])

# =============================================================================
# 4. FINAL RESULTS
# =============================================================================
df_res = pd.DataFrame(results, columns=['Model', 'Train Acc', 'Test Acc', 'Time'])
df_res = df_res.sort_values(by='Test Acc', ascending=False)

print("\n" + "="*60)
print("              FINAL RESULTS (TEST SET)")
print("="*60)
print(df_res.to_string(index=False, float_format="%.4f"))
print("="*60)

fig, ax = plt.subplots(1, 2, figsize=(20, 6))

# Plot 1: Accuracy
acc_melt = df_res.melt(id_vars='Model', value_vars=['Train Acc', 'Test Acc'], var_name='Metric', value_name='Score')
sns.barplot(x='Model', y='Score', hue='Metric', data=acc_melt, ax=ax[0], palette='viridis')
ax[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax[0].set_ylim(0.6, 1.05)
ax[0].grid(axis='y', linestyle='--', alpha=0.7)
ax[0].legend(loc='lower right')
plt.setp(ax[0].get_xticklabels(), rotation=15)

for p in ax[0].patches:
    if p.get_height() > 0:
        ax[0].annotate(f'{p.get_height():.2%}', (p.get_x() + p.get_width()/2, p.get_height()),
                       ha='center', va='bottom', fontsize=9)

# Plot 2: Time
sns.barplot(x='Model', y='Time', data=df_res, ax=ax[1], palette='Reds_r')
ax[1].set_title('Total Time (Training + Prediction)', fontsize=14, fontweight='bold')
ax[1].set_ylabel('Seconds')
ax[1].grid(axis='y', linestyle='--', alpha=0.7)
plt.setp(ax[1].get_xticklabels(), rotation=15)

for p in ax[1].patches:
    ax[1].annotate(f'{p.get_height():.1f}s', (p.get_x() + p.get_width()/2, p.get_height()),
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

*Linear* (In every other experiment the only change is in the hyperparameter values)

In [ ]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import cuml
from cuml.svm import LinearSVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOADING & PREPARING THE DATA
# =============================================================================
print("\n--- Loading CIFAR-10 ---")
(x_train_orig, y_train_orig), (x_test_orig, y_test_orig) = cifar10.load_data()

# Build the classes: Animals (0) vs Vehicles (1)
animals = [2, 3, 4, 5, 6, 7]
vehicles = [0, 1, 8, 9]

# Convert train labels
y_train = np.copy(y_train_orig).flatten()
mask_train_animals = np.isin(y_train_orig.flatten(), animals)
mask_train_vehicles = np.isin(y_train_orig.flatten(), vehicles)
y_train[mask_train_animals] = 0
y_train[mask_train_vehicles] = 1

# Convert test labels
y_test = np.copy(y_test_orig).flatten()
mask_test_animals = np.isin(y_test_orig.flatten(), animals)
mask_test_vehicles = np.isin(y_test_orig.flatten(), vehicles)
y_test[mask_test_animals] = 0
y_test[mask_test_vehicles] = 1

print(f"Training Data: {len(y_train)} images (Full Set)")
print(f"Testing Data:  {len(y_test)} images (Full Set)")

# Flattening: reshape from (N, 32, 32, 3) to (N, 3072) & Normalization
X_train = x_train_orig.reshape(x_train_orig.shape[0], -1).astype('float32') / 255.0
X_test = x_test_orig.reshape(x_test_orig.shape[0], -1).astype('float32') / 255.0

# Scaling: needed so the SVMs converge faster and correctly
print("Applying scaling...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =============================================================================
# 2. GRID SEARCH (LinearSVC)
# =============================================================================
print("\n--- Starting Grid Search on GPU (LinearSVC) ---")
# LinearSVC is used because it is much faster than SVC(kernel='linear')
linear_svc = LinearSVC(penalty='l2', loss='squared_hinge', class_weight='balanced', verbose=False,max_iter=10000)

param_grid = [
    {

        'C': [0.01, 0.1, 1, 5, 10, 50]
    }
]

grid_search = GridSearchCV(
    linear_svc,
    param_grid,
    refit=True,#at the end it retrains on all the data
    verbose=2,
    cv=5,
    return_train_score=True
)

start_time = time.time()
grid_search.fit(X_train_scaled, y_train)
end_time = time.time()

print(f"Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3. MAIN RESULTS
# =============================================================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

# Convert from cupy to numpy
if hasattr(y_pred, 'to_numpy'):
    y_pred = y_pred.to_numpy()
elif hasattr(y_pred, 'get'):
    y_pred = y_pred.get()

print("\n" + "="*60)
print(f" Best Params: {grid_search.best_params_}")
print("="*60)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Animals', 'Vehicles'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Animals', 'Vehicles'], yticklabels=['Animals', 'Vehicles'])
plt.title('Confusion Matrix (LinearSVC Final Model)')
plt.show()

# =============================================================================
# 4. DETAILED PLOTS & TABLES
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)
# Map the C values to simple indices (1, 2, 3...)
# so that the x-axis of the plots is easier to read.
unique_Cs = sorted(results_df['param_C'].unique())
c_map = {val: i+1 for i, val in enumerate(unique_Cs)}
# Sort the results by Validation Accuracy, descending
def get_short_label_linear(row):
    c_idx = c_map[row['param_C']]
    return f"{c_idx}"

results_df['short_label'] = results_df.apply(get_short_label_linear, axis=1)
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)

# --- Print the mapping guide
print("\n" + "="*80)
print("   X-AXIS GUIDE (Index C)")
print("="*80)
print("C indices:")
for val, idx in c_map.items():
    print(f"  {idx} = {val}")
print("="*80 + "\n")

# --- Print table
print("##  Sorted Grid Search Results (LinearSVC)")
print("-" * 80)
print(f"{'Code':<10} | {'Parameters':<35} | {'Train Acc':<12} | {'Val Acc':<12}")
print("-" * 80)

for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100
    params_full = f"C:{row['param_C']}"
    print(f"{row['short_label']:<10} | {params_full:<35} | {train_acc:.2f}%      | {val_acc:.2f}%")
print("\n")

# --- Plots ---
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Accuracy
acc_data = results_df_sorted[['short_label', 'mean_train_score', 'mean_test_score']].melt(
    id_vars='short_label', var_name='Type', value_name='Accuracy'
)
acc_data['Type'] = acc_data['Type'].replace({
    'mean_train_score': 'Train Accuracy', 'mean_test_score': 'Validation Accuracy'
})

sns.barplot(x='short_label', y='Accuracy', hue='Type', data=acc_data, ax=ax[0], palette='viridis',
            order=results_df_sorted['short_label'])

ax[0].set_title('Accuracy Comparison (LinearSVC) - Sorted', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Accuracy (0-1)', fontsize=12)
ax[0].set_xlabel('Index C - From Best to Worst', fontsize=12)
ax[0].set_ylim(0.70, 1.01)
ax[0].legend(title='Metric', loc='lower right')
ax[0].grid(axis='y', linestyle='--', alpha=0.7)

for p in ax[0].patches:
    if p.get_height() > 0:
        ax[0].annotate(f'{p.get_height():.4f}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha = 'center', va = 'bottom', xytext = (0, 5),
                       textcoords = 'offset points', fontsize=8, rotation=90)

# Plot 2: Time
sns.barplot(x='short_label', y='mean_fit_time', data=results_df_sorted, ax=ax[1], palette='Reds_r',
            order=results_df_sorted['short_label'])

ax[1].set_title('Training Time (LinearSVC)', fontsize=14, fontweight='bold')
ax[1].set_ylabel('Seconds', fontsize=12)
ax[1].set_xlabel('Index C - From Best to Worst', fontsize=12)
ax[1].grid(axis='y', linestyle='--', alpha=0.7)

for p in ax[1].patches:
    ax[1].annotate(f'{p.get_height():.3f}s',
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'bottom', fontsize=9)

plt.tight_layout()
plt.show()

RBF

In [ ]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import cuml
from cuml.svm import SVC as GPU_SVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOADING & PREPARING THE DATA
# =============================================================================
print("\n--- Loading CIFAR-10 ---")
(x_train_orig, y_train_orig), (x_test_orig, y_test_orig) = cifar10.load_data()

# Build the classes: Animals (0) vs Vehicles (1)
animals = [2, 3, 4, 5, 6, 7]
vehicles = [0, 1, 8, 9]

# Convert train labels
y_train = np.copy(y_train_orig).flatten()
mask_train_animals = np.isin(y_train_orig.flatten(), animals)
mask_train_vehicles = np.isin(y_train_orig.flatten(), vehicles)
y_train[mask_train_animals] = 0
y_train[mask_train_vehicles] = 1

# Convert test labels
y_test = np.copy(y_test_orig).flatten()
mask_test_animals = np.isin(y_test_orig.flatten(), animals)
mask_test_vehicles = np.isin(y_test_orig.flatten(), vehicles)
y_test[mask_test_animals] = 0
y_test[mask_test_vehicles] = 1

print(f"Training Data: {len(y_train)} images (Full Set)")
print(f"Testing Data:  {len(y_test)} images (Full Set)")

# Flattening : reshape from (N, 32, 32, 3) to (N, 3072) & Normalization
X_train = x_train_orig.reshape(x_train_orig.shape[0], -1).astype('float32') / 255.0
X_test = x_test_orig.reshape(x_test_orig.shape[0], -1).astype('float32') / 255.0

# Scaling: needed so the SVMs converge faster and correctly
print("Applying scaling...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =============================================================================
# 2. GRID SEARCH (RBF KERNEL)
# =============================================================================
print("\n--- Starting Grid Search on GPU (RBF Kernel) ---")

svc = GPU_SVC(kernel='rbf', class_weight='balanced', probability=False, cache_size=2000)



param_grid = [
    {
        'kernel': ['rbf'],

        'C': [1, 5, 10, 50],

        'gamma': ['scale', 'auto',0.0001, 0.001,0.01]
    }
]




grid_search = GridSearchCV(

svc,

param_grid,

refit=True,

verbose=2,

cv=5,

return_train_score=True

)

start_time = time.time()
grid_search.fit(X_train_scaled, y_train)
end_time = time.time()
print(f"Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3. MAIN RESULTS
# =============================================================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

# Convert from cupy to numpy
if hasattr(y_pred, 'to_numpy'):
    y_pred = y_pred.to_numpy()
elif hasattr(y_pred, 'get'):
    y_pred = y_pred.get()

print("\n" + "="*60)
print(f" Best Params: {grid_search.best_params_}")
# Print gamma if it is 'scale'
try:
    if grid_search.best_params_.get('gamma') == 'scale':
        print(f"  Actual Gamma Value used ('scale'): {best_model._gamma:.7f}")
except:
    pass
print("="*60)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Animals', 'Vehicles'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Animals', 'Vehicles'], yticklabels=['Animals', 'Vehicles'])
plt.title('Confusion Matrix (RBF Kernel Final Model)')
plt.show()

# =============================================================================
# 4. DETAILED PLOTS & TABLES
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)

# Map the C values to simple indices (1, 2, 3...)
# so that the x-axis of the plots is easier to read.

unique_Cs = sorted(results_df['param_C'].unique())
c_map = {val: i+1 for i, val in enumerate(unique_Cs)}

unique_Gammas = results_df['param_gamma'].unique()
try:
    unique_Gammas = sorted(unique_Gammas)
except:
    pass
g_map = {val: i+1 for i, val in enumerate(unique_Gammas)}
# Sort the results by Validation Accuracy, descending
def get_short_label_rbf(row):
    c_idx = c_map[row['param_C']]
    g_idx = g_map[row['param_gamma']]
    return f"{c_idx}.{g_idx}"

results_df['short_label'] = results_df.apply(get_short_label_rbf, axis=1)

#  Sort by Validation Accuracy
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)


# --- Print the mapping guide
print("\n" + "="*80)
print("   X-AXIS GUIDE (Index C . Index Gamma)")
print("="*80)
print("C indices:")
for val, idx in c_map.items():
    print(f"  {idx} = {val}")
print("Gamma indices:")
for val, idx in g_map.items():
    print(f"  {idx} = {val}")
print("="*80 + "\n")

# --- Print table with Train & Val Acc (SORTED)
print("##  Sorted Grid Search Results (RBF Kernel)")
print("-" * 80)
print(f"{'Code':<10} | {'Parameters':<35} | {'Train Acc':<12} | {'Val Acc':<12}")
print("-" * 80)

for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100

    params_full = f"C:{row['param_C']}, G:{row['param_gamma']}"
    print(f"{row['short_label']:<10} | {params_full:<35} | {train_acc:.2f}%      | {val_acc:.2f}%")
print("\n")


# --- Create 2 plots (SORTED)
fig, ax = plt.subplots(1, 2, figsize=(20, 7))

# Plot 1: Accuracy (Train vs Validation)
acc_data = results_df_sorted[['short_label', 'mean_train_score', 'mean_test_score']].melt(
    id_vars='short_label',
    var_name='Type',
    value_name='Accuracy'
)
acc_data['Type'] = acc_data['Type'].replace({
    'mean_train_score': 'Train Accuracy',
    'mean_test_score': 'Validation Accuracy'
})

sns.barplot(x='short_label', y='Accuracy', hue='Type', data=acc_data, ax=ax[0], palette='viridis',
            order=results_df_sorted['short_label'])

ax[0].set_title('Accuracy Comparison (RBF Kernel) - Sorted', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Accuracy (0-1)', fontsize=12)
ax[0].set_xlabel('Combination (C.G) - From Best to Worst', fontsize=12)
ax[0].set_ylim(0.70, 1.01)
ax[0].legend(title='Metric', loc='lower right')
ax[0].grid(axis='y', linestyle='--', alpha=0.7)

# Labels
for p in ax[0].patches:
    if p.get_height() > 0.90:
        ax[0].annotate(f'{p.get_height():.3f}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha = 'center', va = 'bottom', xytext = (0, 5),
                       textcoords = 'offset points', fontsize=7, rotation=90)


# Plot 2: Time
sns.barplot(x='short_label', y='mean_fit_time', data=results_df_sorted, ax=ax[1], palette='Reds_r',
            order=results_df_sorted['short_label'], hue='short_label', legend=False)

ax[1].set_title('Training Time (RBF Kernel) - Sorted', fontsize=14, fontweight='bold')
ax[1].set_ylabel('Seconds', fontsize=12)
ax[1].set_xlabel('Combination (C.G) - From Best to Worst', fontsize=12) # Updated label
ax[1].grid(axis='y', linestyle='--', alpha=0.7)

for p in ax[1].patches:
    ax[1].annotate(f'{p.get_height():.1f}s',
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'bottom', fontsize=8)

plt.tight_layout()
plt.show()

POLY

In [ ]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import cuml
from cuml.svm import SVC as GPU_SVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOADING & PREPARING THE DATA
# =============================================================================
print("\n--- Loading CIFAR-10 ---")
(x_train_orig, y_train_orig), (x_test_orig, y_test_orig) = cifar10.load_data()

# Build the classes: Animals (0) vs Vehicles (1)
animals = [2, 3, 4, 5, 6, 7]
vehicles = [0, 1, 8, 9]

# Convert train labels
y_train = np.copy(y_train_orig).flatten()
mask_train_animals = np.isin(y_train_orig.flatten(), animals)
mask_train_vehicles = np.isin(y_train_orig.flatten(), vehicles)
y_train[mask_train_animals] = 0
y_train[mask_train_vehicles] = 1

# Convert test labels
y_test = np.copy(y_test_orig).flatten()
mask_test_animals = np.isin(y_test_orig.flatten(), animals)
mask_test_vehicles = np.isin(y_test_orig.flatten(), vehicles)
y_test[mask_test_animals] = 0
y_test[mask_test_vehicles] = 1

print(f"Training Data: {len(y_train)} images (Full Set)")
print(f"Testing Data:  {len(y_test)} images (Full Set)")

# Flattening: reshape from (N, 32, 32, 3) to (N, 3072) & Normalization
X_train = x_train_orig.reshape(x_train_orig.shape[0], -1).astype('float32') / 255.0
X_test = x_test_orig.reshape(x_test_orig.shape[0], -1).astype('float32') / 255.0

# Scaling: needed so the SVMs converge faster and correctly
print("Applying scaling...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =============================================================================
# 2. GRID SEARCH (POLY KERNEL)
# =============================================================================
print("\n--- Starting Grid Search on GPU (Poly Kernel) ---")

POLU = GPU_SVC(kernel='poly', class_weight='balanced', probability=False, cache_size=2000)





param_grid = [
    {
        'kernel': ['poly'],

        'degree': [2,3,4],
        'coef0': [1],

        'C': [1, 5, 10],

        'gamma': ['scale', 0.0001, 0.001]
    }
]


grid_search = GridSearchCV(

POLU,

param_grid,

refit=True,

verbose=2,

cv=5,

return_train_score=True

)

start_time = time.time()
grid_search.fit(X_train_scaled, y_train)
end_time = time.time()
print(f"Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3. MAIN RESULTS
# =============================================================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

# Convert from cupy to numpy (automatic with cuML)
if hasattr(y_pred, 'to_numpy'):
    y_pred = y_pred.to_numpy()
elif hasattr(y_pred, 'get'):
    y_pred = y_pred.get()

print("\n" + "="*60)
print(f" Best Params: {grid_search.best_params_}")
# Print gamma if it is 'scale'
try:
    if grid_search.best_params_.get('gamma') == 'scale':
        print(f"  Actual Gamma Value used ('scale'): {best_model._gamma:.7f}")
except:
    pass
print("="*60)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Animals', 'Vehicles'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Animals', 'Vehicles'], yticklabels=['Animals', 'Vehicles'])
plt.title('Confusion Matrix (Poly Kernel Final Model)')
plt.show()

# =============================================================================
# 4. DETAILED PLOTS & TABLES
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)

# Map the C values to simple indices (1, 2, 3...)
# so that the x-axis of the plots is easier to read.
unique_degrees = sorted(results_df['param_degree'].unique())
d_map = {val: i+1 for i, val in enumerate(unique_degrees)}

unique_Cs = sorted(results_df['param_C'].unique())
c_map = {val: i+1 for i, val in enumerate(unique_Cs)}

unique_Gammas = results_df['param_gamma'].unique()
try:
    unique_Gammas = sorted(unique_Gammas)
except:
    pass
g_map = {val: i+1 for i, val in enumerate(unique_Gammas)}

def get_short_label_poly(row):
    d_idx = d_map[row['param_degree']]
    c_idx = c_map[row['param_C']]
    g_idx = g_map[row['param_gamma']]
    return f"{d_idx}.{c_idx}.{g_idx}"

results_df['short_label'] = results_df.apply(get_short_label_poly, axis=1)

# Sort by Validation Accuracy
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)


# --- Print the mapping guide
print("\n" + "="*80)
print("   X-AXIS GUIDE (Index Degree . Index C . Index Gamma)")
print("="*80)
print("Degree indices:")
for val, idx in d_map.items():
    print(f"  {idx} = {val}")
print("C indices:")
for val, idx in c_map.items():
    print(f"  {idx} = {val}")
print("Gamma indices:")
for val, idx in g_map.items():
    print(f"  {idx} = {val}")
print("="*80 + "\n")

# --- Print table with Train & Val Acc (SORTED)
print("## Sorted Grid Search Results (Poly Kernel)")
print("-" * 80)
print(f"{'Code':<10} | {'Parameters':<35} | {'Train Acc':<12} | {'Val Acc':<12}")
print("-" * 80)

for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100
    params_full = f"D:{row['param_degree']}, C:{row['param_C']}, G:{row['param_gamma']}, C0:{row['param_coef0']}"
    print(f"{row['short_label']:<10} | {params_full:<35} | {train_acc:.2f}%      | {val_acc:.2f}%")
print("\n")


# --- Create 2 plots (SORTED)
fig, ax = plt.subplots(1, 2, figsize=(20, 7))

# Plot 1: Accuracy (Train vs Validation)
acc_data = results_df_sorted[['short_label', 'mean_train_score', 'mean_test_score']].melt(
    id_vars='short_label',
    var_name='Type',
    value_name='Accuracy'
)
acc_data['Type'] = acc_data['Type'].replace({
    'mean_train_score': 'Train Accuracy',
    'mean_test_score': 'Validation Accuracy'
})

sns.barplot(x='short_label', y='Accuracy', hue='Type', data=acc_data, ax=ax[0], palette='viridis',
            order=results_df_sorted['short_label'])

ax[0].set_title('Accuracy Comparison (Poly Kernel) - Sorted', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Accuracy (0-1)', fontsize=12)
ax[0].set_xlabel('Combination (D.C.G) - From Best to Worst', fontsize=12)
ax[0].set_ylim(0.70, 1.01)
ax[0].legend(title='Metric', loc='lower right')
ax[0].grid(axis='y', linestyle='--', alpha=0.7)

# Labels
for p in ax[0].patches:
    if p.get_height() > 0.90:
        ax[0].annotate(f'{p.get_height():.3f}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha = 'center', va = 'bottom', xytext = (0, 5),
                       textcoords = 'offset points', fontsize=7, rotation=90)


# Plot 2: Time
sns.barplot(x='short_label', y='mean_fit_time', data=results_df_sorted, ax=ax[1], palette='Reds_r',
            order=results_df_sorted['short_label'])

ax[1].set_title('Training Time (Poly Kernel) - Sorted', fontsize=14, fontweight='bold')
ax[1].set_ylabel('Seconds', fontsize=12)
ax[1].set_xlabel('Combination (D.C.G) - From Best to Worst', fontsize=12)
ax[1].grid(axis='y', linestyle='--', alpha=0.7)

for p in ax[1].patches:
    ax[1].annotate(f'{p.get_height():.1f}s',
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'bottom', fontsize=8)

plt.tight_layout()
plt.show()